In [1]:
import subprocess, os

NORM_URL  = "http://ml.cs.tsinghua.edu.cn/~lingxuan/rdt2/umi_normalizer_wo_downsample_indentity_rot.pt"
NORM_PATH = "/home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt"

if not os.path.exists(NORM_PATH):
    print("Downloading official normalizer...")
    result = subprocess.run(
        ['wget', '-O', NORM_PATH, NORM_URL],
        capture_output=True, text=True
    )
    print(result.stdout[-500:])
    print(result.stderr[-500:])

size = os.path.getsize(NORM_PATH) / 1024
print(f"✅ Normalizer ready: {size:.1f} KB")

✅ Normalizer ready: 27.5 KB


In [2]:
import os, sys, numpy as np, torch
from PIL import Image

RDT2_DIR    = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
CHECKPOINT  = "/home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-800"
NORM_PATH   = "/home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt"
DATASET     = "/home/rishabh/Downloads/umi-pipeline-training/replay_buffer.zarr"
INSTRUCTION = "Pick up the marker and place it in the box."

# Add ALL required paths
sys.path.insert(0, RDT2_DIR)
sys.path.insert(0, os.path.join(RDT2_DIR, 'vqvae'))
sys.path.insert(0, os.path.join(RDT2_DIR, 'models'))

os.chdir(RDT2_DIR)  # ← critical: must run FROM RDT2 directory

os.environ['TRANSFORMERS_ATTN_IMPLEMENTATION'] = 'eager'
os.environ['PYTORCH_CUDA_ALLOC_CONF']          = 'expandable_segments:True'

# Verify paths exist
for p in [RDT2_DIR, CHECKPOINT, NORM_PATH, DATASET]:
    exists = "✅" if os.path.exists(p) else "❌"
    print(f"{exists} {p}")

print("\n✅ Setup done")
print(f"   cwd: {os.getcwd()}")
print(f"   sys.path[0]: {sys.path[0]}")

✅ /home/rishabh/Downloads/umi-pipeline-training/RDT2
❌ /home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-800
✅ /home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt
✅ /home/rishabh/Downloads/umi-pipeline-training/replay_buffer.zarr

✅ Setup done
   cwd: /home/rishabh/Downloads/umi-pipeline-training/RDT2
   sys.path[0]: /home/rishabh/Downloads/umi-pipeline-training/RDT2/models


In [3]:
import torch, sys, os
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from peft import PeftModel
from models.multivqvae import MultiVQVAE
from models.normalizer.normalizer import LinearNormalizer

device = "cuda:0"

print("[1/4] Loading base model...")
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "robotics-diffusion-transformer/RDT2-VQ",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    device_map=device
)

print("[2/4] Loading YOUR fine-tuned adapter...")
model = PeftModel.from_pretrained(base, CHECKPOINT)
model.eval()

print("[3/4] Loading processor...")
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct", use_fast=True
)

print("[4/4] Loading VAE...")
vae = MultiVQVAE.from_pretrained(
    "robotics-diffusion-transformer/RVQActionTokenizer"
).eval().to(device=device, dtype=torch.float32)

# ── Load normalizer — try all methods ─────────────────────────
print("Loading official normalizer...")
norm_loaded = False

# Method 1: .load()
try:
    normalizer = LinearNormalizer.load(NORM_PATH)
    print("✅ LinearNormalizer.load() worked")
    norm_loaded = True
except Exception as e1:
    print(f"   load() failed: {e1}")

# Method 2: torch.load directly
if not norm_loaded:
    try:
        normalizer = torch.load(NORM_PATH, map_location='cpu')
        print(f"✅ torch.load() worked, type: {type(normalizer)}")
        norm_loaded = True
    except Exception as e2:
        print(f"   torch.load() failed: {e2}")

# Method 3: instantiate then load state dict
if not norm_loaded:
    try:
        data = torch.load(NORM_PATH, map_location='cpu',
                          weights_only=False)
        print(f"✅ Loaded data type: {type(data)}")
        print(f"   Keys: {data.keys() if hasattr(data, 'keys') else 'no keys'}")
        normalizer = data
        norm_loaded = True
    except Exception as e3:
        print(f"   method 3 failed: {e3}")

print(f"\nNormalizer type: {type(normalizer)}")

# Check if it has 'action' key
try:
    print(f"normalizer['action']: {type(normalizer['action'])}")
except Exception as e:
    print(f"No 'action' key: {e}")
    print(f"Dir: {[a for a in dir(normalizer) if not a.startswith('_')]}")

valid_action_id_length = (
    vae.pos_id_len + vae.rot_id_len + vae.grip_id_len
)

print(f"\n✅ ALL LOADED!")
print(f"   valid_action_id_length: {valid_action_id_length}")

/home/rishabh/Downloads/umi-pipeline-training/umi_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[1/4] Loading base model...


Loading checkpoint shards: 100%|██████████| 4/4 [00:01<00:00,  2.39it/s]


[2/4] Loading YOUR fine-tuned adapter...


ValueError: Can't find 'adapter_config.json' at '/home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-800'

In [ ]:
import zarr, numpy as np, torch
from PIL import Image

root    = zarr.open(DATASET, mode='r')
img_arr = np.array(root['data']['camera0_rgb'][500], dtype=np.uint8)

# Resize to 384x384
img_384 = np.array(
    Image.fromarray(img_arr).resize((384, 384)),
    dtype=np.uint8
)  # (384, 384, 3)

# Convert to tensor — shape must be (N, H, W, 3)
# utils.py line: image = data_dict["obs"]["camera0_rgb"][-1]
# So we need (N, 384, 384, 3) where [-1] gives (384, 384, 3)
img_tensor = torch.from_numpy(img_384).unsqueeze(0)  # (1, 384, 384, 3)

print(f"✅ Image tensor shape: {img_tensor.shape}")
print(f"   dtype: {img_tensor.dtype}")
print(f"   [-1] shape: {img_tensor[-1].shape}")  # what utils.py sees

✅ Image tensor shape: torch.Size([1, 384, 384, 3])
   dtype: torch.uint8
   [-1] shape: torch.Size([384, 384, 3])


In [ ]:
import numpy as np, torch
from utils import batch_predict_action

print(f"🧠 Running official inference (single camera)...")
print(f"   Instruction: '{INSTRUCTION}'")

result = batch_predict_action(
    model, processor, vae, normalizer,
    examples=[{
        "obs": {"camera0_rgb": img_tensor},
        "meta": {"num_camera": 1}
    }],
    valid_action_id_length=valid_action_id_length,
    apply_jpeg_compression=True,
    instruction=INSTRUCTION
)

action_chunk = result["action_pred"][0].cpu().detach()  # ← detach added
print(f"✅ Action chunk shape: {action_chunk.shape}")

# Rescale gripper
for robot_idx in range(2):
    action_chunk[:, robot_idx * 10 + 9] = (
        action_chunk[:, robot_idx * 10 + 9] / 0.088 * 0.1
    )

# RIGHT ARM first timestep
right_pos   = action_chunk[0, 0:3].numpy()
right_rot6d = action_chunk[0, 3:9].numpy()
right_grip  = float(action_chunk[0, 9].numpy())

print(f"\nRIGHT ARM step 0:")
print(f"  pos:     {np.round(right_pos, 4)} m")
print(f"  rot(6D): {np.round(right_rot6d, 4)}")
print(f"  gripper: {right_grip:.4f} m")

def rot6d_to_euler(rot6d):
    a1 = rot6d[:3]
    a2 = rot6d[3:]
    b1 = a1 / (np.linalg.norm(a1) + 1e-8)
    b2 = a2 - np.dot(b1, a2) * b1
    b2 = b2 / (np.linalg.norm(b2) + 1e-8)
    b3 = np.cross(b1, b2)
    R  = np.stack([b1, b2, b3], axis=-1)
    sy = np.sqrt(R[0,0]**2 + R[1,0]**2)
    if sy > 1e-6:
        rx = np.degrees(np.arctan2( R[2,1], R[2,2]))
        ry = np.degrees(np.arctan2(-R[2,0], sy))
        rz = np.degrees(np.arctan2( R[1,0], R[0,0]))
    else:
        rx = np.degrees(np.arctan2(-R[1,2], R[1,1]))
        ry = np.degrees(np.arctan2(-R[2,0], sy))
        rz = 0.0
    return np.array([rx, ry, rz])

euler    = rot6d_to_euler(right_rot6d)
grip_pct = int(np.clip(right_grip / 0.1 * 100, 0, 100))

m750 = [
    round(float(right_pos[0]) * 1000, 1),
    round(float(right_pos[1]) * 1000, 1),
    round(float(right_pos[2]) * 1000, 1),
    round(float(euler[0]), 1),
    round(float(euler[1]), 1),
    round(float(euler[2]), 1),
]

print("\n" + "=" * 55)
print("✅ FINAL ACTION (official pipeline):")
print("=" * 55)
print(f"  x:       {right_pos[0]:.4f} m → {right_pos[0]*1000:.1f} mm")
print(f"  y:       {right_pos[1]:.4f} m → {right_pos[1]*1000:.1f} mm")
print(f"  z:       {right_pos[2]:.4f} m → {right_pos[2]*1000:.1f} mm")
print(f"  rx:      {euler[0]:.2f} deg")
print(f"  ry:      {euler[1]:.2f} deg")
print(f"  rz:      {euler[2]:.2f} deg")
print(f"  gripper: {right_grip:.4f} m → {grip_pct}/100")
print("=" * 55)
print(f"\n🤖 M750 COMMAND:")
print(f"   send_coords({m750}, speed=15, mode=1)")
print(f"   set_gripper_value({grip_pct}, speed=50)")

last_m750 = m750
last_grip = grip_pct
print("\n✅ Ready for robot!")

🧠 Running official inference (single camera)...
   Instruction: 'Pick up the marker and place it in the box.'
✅ Action chunk shape: torch.Size([24, 20])

RIGHT ARM step 0:
  pos:     [ 1.e-04  3.e-04 -1.e-04] m
  rot(6D): [ 2.8233606e+03  3.0739999e-01 -1.0652000e+00 -1.2707700e+02
  3.1090588e+03 -9.8439997e-01]
  gripper: 0.0701 m

✅ FINAL ACTION (official pipeline):
  x:       0.0001 m → 0.1 mm
  y:       0.0003 m → 0.3 mm
  z:       -0.0001 m → -0.1 mm
  rx:      -0.02 deg
  ry:      0.02 deg
  rz:      0.01 deg
  gripper: 0.0701 m → 70/100

🤖 M750 COMMAND:
   send_coords([0.1, 0.3, -0.1, -0.0, 0.0, 0.0], speed=15, mode=1)
   set_gripper_value(70, speed=50)

✅ Ready for robot!


In [ ]:
# Test all available checkpoints to find the best one
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel
import numpy as np
from PIL import Image

BASE_DIR = "/home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned"
CHECKPOINTS = [
    "checkpoint-4600",
    "checkpoint-4800", 
    "checkpoint-5000"
]

# Load dataset image for testing
import zarr
DATASET_PATH = "/home/rishabh/Downloads/umi-pipeline-training/replay_buffer.zarr"
root = zarr.open(DATASET_PATH, mode='r')
img_arr = np.array(root['data']['camera0_rgb'][500], dtype=np.uint8)
test_img = Image.fromarray(img_arr).resize((384, 384))

print("="*70)
print("TESTING ALL CHECKPOINTS")
print("="*70)

results = {}

for checkpoint_name in CHECKPOINTS:
    checkpoint_path = f"{BASE_DIR}/{checkpoint_name}"
    
    print(f"\n🔍 Testing {checkpoint_name}...")
    
    try:
        # Load model
        base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            "robotics-diffusion-transformer/RDT2-VQ",
            torch_dtype=torch.bfloat16,
            attn_implementation="eager",
            device_map="cuda"
        )
        
        model = PeftModel.from_pretrained(base, checkpoint_path)
        model.eval()
        
        processor = AutoProcessor.from_pretrained(
            "Qwen/Qwen2.5-VL-7B-Instruct", use_fast=True
        )
        
        # Run quick inference test
        from models.multivqvae import MultiVQVAE
        vae = MultiVQVAE.from_pretrained(
            "robotics-diffusion-transformer/RVQActionTokenizer"
        ).cuda().eval()
        
        # Simple prediction
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": test_img},
                {"type": "text", "text": "Pick up the marker and place it in the box."}
            ]
        }]
        
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        
        inputs = processor(
            text=[text], images=[test_img], return_tensors="pt"
        ).to("cuda", torch.bfloat16)
        
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)
        
        # Get prediction stats
        input_len = inputs["input_ids"].shape[1]
        generated = output_ids[:, input_len:].cpu()
        tokens = generated[0].numpy()
        
        # Quick decode
        SPECIAL = {151643, 151644, 151645, 151646, 151650, 151651,
                   151652, 151653, 151654, 151655, 151656}
        clean = [t for t in tokens.tolist() if t not in SPECIAL]
        
        BASE = 151000
        total = 27
        remapped = [max(0, t - BASE) for t in clean[:total]]
        ids_t = torch.tensor(remapped, dtype=torch.long).unsqueeze(0)
        
        vae_cpu = vae.cpu()
        with torch.no_grad():
            result = vae_cpu.decode(ids_t)
        
        # Get first timestep action
        action = result[0, 0].float().numpy()[:7]
        
        results[checkpoint_name] = {
            'tokens': len(clean),
            'action': action,
            'pos_range': (action[:3].min(), action[:3].max()),
            'grip': action[6]
        }
        
        print(f"  ✅ Position: [{action[0]:.4f}, {action[1]:.4f}, {action[2]:.4f}]")
        print(f"  ✅ Gripper: {action[6]:.4f}")
        
        # Cleanup
        del model, base, inputs
        torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"  ❌ Failed: {e}")
        results[checkpoint_name] = None

# Compare results
print("\n" + "="*70)
print("COMPARISON SUMMARY")
print("="*70)

for name, result in results.items():
    if result:
        pos_range = result['pos_range'][1] - result['pos_range'][0]
        print(f"\n{name}:")
        print(f"  Position range: {pos_range:.6f}")
        print(f"  Action: {np.round(result['action'], 4)}")
    else:
        print(f"\n{name}: FAILED")

print("\n" + "="*70)
print("RECOMMENDATION:")

# Find checkpoint with most diversity
if any(results.values()):
    best = max(
        [(k, v) for k, v in results.items() if v],
        key=lambda x: (x[1]['pos_range'][1] - x[1]['pos_range'][0])
    )
    print(f"Use: {best[0]} (has most position diversity)")
else:
    print("ALL CHECKPOINTS FAILED - Need to retrain!")

print("="*70)


TESTING ALL CHECKPOINTS

🔍 Testing checkpoint-4600...


Loading checkpoint shards:  25%|██▌       | 1/4 [00:00<00:02,  1.30it/s]


  ❌ Failed: CUDA out of memory. Tried to allocate 130.00 MiB. GPU 0 has a total capacity of 23.52 GiB of which 130.06 MiB is free. Including non-PyTorch memory, this process has 23.37 GiB memory in use. Of the allocated memory 22.98 GiB is allocated by PyTorch, and 7.69 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

🔍 Testing checkpoint-4800...


Loading checkpoint shards:  25%|██▌       | 1/4 [00:00<00:02,  1.31it/s]


  ❌ Failed: CUDA out of memory. Tried to allocate 130.00 MiB. GPU 0 has a total capacity of 23.52 GiB of which 130.06 MiB is free. Including non-PyTorch memory, this process has 23.37 GiB memory in use. Of the allocated memory 22.98 GiB is allocated by PyTorch, and 7.69 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

🔍 Testing checkpoint-5000...


Loading checkpoint shards:  25%|██▌       | 1/4 [00:00<00:02,  1.31it/s]

  ❌ Failed: CUDA out of memory. Tried to allocate 130.00 MiB. GPU 0 has a total capacity of 23.52 GiB of which 130.06 MiB is free. Including non-PyTorch memory, this process has 23.37 GiB memory in use. Of the allocated memory 22.98 GiB is allocated by PyTorch, and 7.69 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

COMPARISON SUMMARY

checkpoint-4600: FAILED

checkpoint-4800: FAILED

checkpoint-5000: FAILED

RECOMMENDATION:
ALL CHECKPOINTS FAILED - Need to retrain!


: 